# RQ3 PLOTS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

def alphanumeric_sort_key(s):
    """Split the string into a list of strings and integers for sorting."""
    return [int(text) if text.isdigit() else text for text in re.split('(\d+)', s)]

def plot_metrics_from_csv(file_path, group_by_column, filter_substring, title_prefix, metrics_to_plot):
    # Read the CSV file into a DataFrame
    df = pd.read_csv(file_path)
    
    # Filter the DataFrame to include only rows where 'Model Name' contains the filter_substring
    filtered_df = df[df['Model Name'].str.contains(filter_substring)]
    
    # Apply additional filtering based on group_by_column
    if group_by_column == 'issueArea':
        filtered_df = filtered_df[filtered_df['issueArea'].isin([1, 2, 3, 8, 9])]
    elif group_by_column == 'respondentType':
        filtered_df = filtered_df[filtered_df['respondentType'].isin([1, 2, 3, 4])]
    
    # Drop all columns except the ones we want to plot
    columns_to_keep = ['Model Name', group_by_column] + metrics_to_plot
    filtered_df = filtered_df[columns_to_keep]
    
    # Sort the DataFrame by 'Model Name' using the custom sorting key
    sorted_df = filtered_df.sort_values(by='Model Name', key=lambda col: col.map(alphanumeric_sort_key))
    
    # Group the DataFrame by the specified column
    grouped = sorted_df.groupby(group_by_column)
    
    # Iterate over each group
    for group_name, group in grouped:
        plt.figure(figsize=(12, 8))
        for column in group.columns[2:]:  # Skip 'Model Name' and the group_by_column
            y = group[column]
            plt.plot(group['Model Name'], y, marker='o', label=column)
            
            # Highlight the maximum value
            max_value = y.max()
            max_indices = y[y == max_value].index
            plt.scatter(group['Model Name'][max_indices], y[max_indices], color='black', s=100, zorder=5, label=f'{column} max')

            # Highlight the minimum value
            min_value = y.min()
            min_indices = y[y == min_value].index
            plt.scatter(group['Model Name'][min_indices], y[min_indices], color='blue', s=100, zorder=5, label=f'{column} min')
        
        plt.title(f"{title_prefix} - {group_by_column} {group_name}")
        plt.xlabel('Model Name')
        plt.ylabel('Score')
        plt.legend()
        plt.grid(True)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

def iterate_and_plot(file_names, filter_substrings, metrics):
    for file_name in file_names:
        for filter_substring in filter_substrings:
            for metric in metrics:
                if metric.startswith('unieval'):
                    metrics_to_plot = ['unieval_coherence', 'unieval_fluency']
                else:
                    metrics_to_plot = [metric]
                
                plot_metrics_from_csv(
                    f'/home/mostah/workspace/fairness/scores/grouped average scores/{file_name}.csv',
                    file_name,
                    filter_substring,
                    f'Filtered Metrics Plot for {metric}',
                    metrics_to_plot
                )

# Lists for iteration
file_names = ['decisionDirection', 'issueArea', 'partyWinning', 'respondentType', 'voteDistribution']
filter_substrings = ['temp', 'top_p', 'top_k']
metrics = ['rougeL', 'align_score', 'bert_f1', 'unieval_']

# Call the iterate_and_plot function
iterate_and_plot(file_names, filter_substrings, metrics)
